In [ ]:
from vistiq.io import ImageLoader, ImageLoaderConfig, ImageWriter, ImageWriterConfig
from vistiq.utils import check_device, ArrayIteratorConfig
from vistiq.preprocess import PreprocessFlowConfig, FuncProcessorConfig, PeriodicDenoise, PeriodicDenoiseConfig
from vistiq.segment import MicroSAMSegmenterConfig, MicroSAMSegmenter, SegmentationFlowConfig, SegmentationFlow

import logging
import stackview
import numpy as np

# Configure logger and check availability of accelerator

In [ ]:
import vistiq
logger = logging.getLogger(vistiq.__name__)

logger.info(f"Available Torch accelerators: {check_device()}")

# Load image

In [ ]:
data_dir = "/project/skyrmion/data/"
fname = f"{data_dir}/2026-07-22-patches/Camera Ceta 1149 0001 Ceta.tif"
fname = f"{data_dir}/Si3N4_0001_multislice.emd"

ch = 1
icfg = ImageLoaderConfig(squeeze=True, substack=f"S:{ch}", rename_channel={"Channel:0:0":f"channel {ch}"})

img, metadata = ImageLoader(icfg).run(fname)
metadata

In [ ]:
stackview.slice(img)

# Preprocess

In [ ]:
pdcfg = PeriodicDenoiseConfig(
    min_distance=50,
    threshold=0.001,
    radius=15,
    mask_angle=[139.0],
    angle_tolerance=1.0,
    mode="degree",
    iterator_config=ArrayIteratorConfig(slice_def=(-2, -1)),
)

denoised,_ = PeriodicDenoise(pdcfg).run(img, metadata=metadata)

In [ ]:
stackview.curtain(img, denoised, continuous_update=False)

# Segment

In [ ]:
embedding_path = "embeddings"

mcfg = MicroSAMSegmenterConfig(
    model_type="vit_l_lm", # "vit_l_em_organelles", # 
    embedding_path=embedding_path,
    iterator_config=ArrayIteratorConfig(slice_def=()),
)

scfg = SegmentationFlowConfig(
    segmenter=mcfg,
)

labels = SegmentationFlow(scfg).run(img, metadata=metadata)
labels_denoised = SegmentationFlow(scfg).run(denoised, metadata=metadata)

In [ ]:
stackview.slice(np.concatenate([labels_denoised], axis=-1))
#img.dtype, denoised.dtype, labels.dtype, labels_denoised.dtype

# Save images

In [ ]:
iwcfg = ImageWriterConfig(format="tif")
path = ".".join(fname.split(".")[:-1]) + "-denoised.tif"
print (path)
ImageWriter(iwcfg).run(denoised, path=path, metadata=metadata)

# Napari Image Viewer

In [ ]:
import napari

In [ ]:
viewer = napari.Viewer()

In [ ]:
viewer.add_image(img, name='Image', colormap='cyan')
viewer.add_image(denoised, name='Denoised', colormap='gray')